# ViziGenesis V2 — EDA & Feature Analysis

This notebook demonstrates the V2 feature engineering pipeline,
regime detection, and data exploration.

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Fetch Data

In [ ]:
from backend.v2.panel_data import fetch_stock_data
from backend.v2.config import PILOT_TICKERS

# Fetch one stock for demonstration
symbol = 'AAPL'
df_raw = fetch_stock_data(symbol, period='5y')
print(f"{symbol}: {len(df_raw)} rows from {df_raw.index[0].date()} to {df_raw.index[-1].date()}")
df_raw.tail()

## 2. Feature Engineering (59 features)

In [ ]:
from backend.v2.features import build_full_features, generate_targets, generate_regime_labels

df_feat = build_full_features(df_raw, symbol)
df_feat = generate_targets(df_feat)
print(f"Features: {df_feat.shape[1]} columns, {len(df_feat)} rows")
print(f"\nFeature groups:")
print(df_feat.columns.tolist())

In [ ]:
# Check feature coverage (NaN %)
nan_pct = (df_feat.isnull().sum() / len(df_feat) * 100).sort_values(ascending=False)
print("Top 15 features by NaN %:")
print(nan_pct.head(15).to_string())

## 3. Regime Detection

In [ ]:
from backend.v2.regime import detect_regime, compute_regime_statistics, get_current_regime

regime_labels = detect_regime(df_feat, method='rules')
df_feat['regime'] = regime_labels

# Regime distribution
print("Regime distribution:")
print(regime_labels.value_counts())
print(f"\nCurrent regime: {get_current_regime(df_feat)}")

In [ ]:
# Plot price colored by regime
fig, ax = plt.subplots(figsize=(16, 6))
colors = {'bull': 'green', 'bear': 'red', 'sideways': 'gray'}
for regime in ['bull', 'bear', 'sideways']:
    mask = df_feat['regime'] == regime
    ax.scatter(df_feat.index[mask], df_feat['Close'][mask], 
               c=colors[regime], s=3, alpha=0.6, label=regime)
ax.set_title(f'{symbol} Price Colored by Regime')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Feature Correlation

In [ ]:
# Correlation of features with next-day direction
from backend.v2.config import V2_FEATURE_COLS

available_cols = [c for c in V2_FEATURE_COLS if c in df_feat.columns]
corr_with_dir = df_feat[available_cols + ['Direction']].corr()['Direction'].drop('Direction').sort_values()

fig, ax = plt.subplots(figsize=(10, max(8, len(corr_with_dir) * 0.2)))
corr_with_dir.plot(kind='barh', ax=ax, color=['red' if x < 0 else 'green' for x in corr_with_dir])
ax.set_title('Feature Correlation with Next-Day Direction')
ax.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

## 5. Per-Regime Feature Statistics

In [ ]:
stats = compute_regime_statistics(df_feat)
for regime, s in stats.items():
    print(f"\n{regime.upper()}:")
    for k, v in s.items():
        print(f"  {k}: {v}")

## 6. Panel Data Overview

In [ ]:
from backend.v2.panel_data import build_panel_dataset

# Build panel for pilot tickers (this may take a minute)
panel = build_panel_dataset(list(PILOT_TICKERS), period='2y')

if panel:
    print(f"Panel X shape: {panel['X'].shape}")
    print(f"Panel y_direction: {panel['y_direction'].shape}")
    print(f"Unique stocks: {len(set(panel['stock_ids']))}")
    print(f"Ticker map: {panel['ticker_map']}")
else:
    print("Could not build panel dataset.")